[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/4chan_thread.ipynb)

# 4chan API: thread

Read one complete thread, the OP and every reply, with `/{board}/thread/{op_id}.json`. This is the only endpoint that returns all replies. The example uses the `/g/` sticky thread, which is written by moderators and never expires.

The 4chan API is read-only JSON. It needs no account and no key. The only
dependency is `requests`, which Google Colab has preinstalled.

Rules from the [API documentation](https://github.com/4chan/4chan-API):
at most one request per second, poll a thread no more often than every 10
seconds, and disclose 4chan as the source of anything you publish from it.
Boards can contain offensive and not-safe-for-work content. Read the
"Research considerations" section on the topic page before collecting.

In [1]:
import requests

In [2]:
api_url = "https://a.4cdn.org/{board}/thread/{op_id}.json"
resp = requests.get(api_url.format(board="g", op_id=105076684))
resp.status_code

200

A `404` means the thread no longer exists. Threads are deleted some time after they are archived, so check the status code before calling `.json()`.

In [3]:
posts = resp.json()["posts"]
len(posts)

4

In [4]:
posts[0]

{'no': 105076684,
 'sticky': 1,
 'closed': 1,
 'now': '04/25/25(Fri)16:24:10',
 'name': 'Anonymous',
 'com': 'This board is for the discussion of technology and related topics.<br>\r\n<br>\r\nReminder that instigating OR participating in flame/brand wars will result in a ban.<br>\r\nTech support threads should be posted to <a href="https://boards.4chan.org/wsr/"><a href="//boards.4chan.org/wsr/" class="quotelink">&gt;&gt;&gt;/wsr/</a></a><br>\r\nCryptocurrency discussion belongs on <a href="https://boards.4chan.org/biz/"><a href="//boards.4chan.org/biz/" class="quotelink">&gt;&gt;&gt;/biz/</a></a><br>\r\n<br>\r\nTo use the Code tag, book-end your body of code with: [code] and [/code]<br>\r\n<br>\r\nThe /g/ Wiki: <a href="https://igwiki.lyci.de/">https://igwiki.lyci.de/</a>',
 'filename': 'sticky btfo',
 'ext': '.png',
 'w': 535,
 'h': 420,
 'tn_w': 250,
 'tn_h': 196,
 'tim': 1745612650141704,
 'time': 1745612650,
 'md5': 'zuZHMJMYYp5WY7vM397nWQ==',
 'fsize': 301273,
 'resto': 0,
 'capc

The fields you will use:

| Field | Meaning |
|---|---|
| `no` | Post number |
| `resto` | Thread ID. 0 for the OP, the OP's `no` for every reply |
| `time` | Unix timestamp |
| `name` | Poster name. Almost always `Anonymous` |
| `trip`, `id` | Tripcode and poster ID. Only present on some boards and posts |
| `com` | Post body as HTML. Quote links look like `<a href="#p123" class="quotelink">&gt;&gt;123</a>` |
| `filename`, `ext`, `tim`, `w`, `h`, `md5` | The attached image, if any |
| `replies`, `images` | OP only. Counts for the thread |
| `archived`, `archived_on` | OP only. Set once the thread has expired |

In [5]:
for post in posts:
    print(post["no"], post["resto"], post["time"], post.get("ext"))

105076684 0 1745612650 .png
105076685 105076684 1745612666 .png
105076689 105076684 1745612673 .gif
105076692 105076684 1745612680 .png


## Who replies to whom

4chan has no reply button. A reply quotes another post by writing `>>123` in
its body, which the site renders as a link. In the JSON the body is HTML, so
the two `>` characters arrive as `&gt;&gt;`. A regular expression recovers the
post numbers, and from them you can build the reply tree of a thread.

The sticky has no quote links, so the cell below picks the busiest thread on
page 1 of the catalog and prints post numbers only.

In [6]:
import re

def quoted_posts(post):
    """Post numbers that this post replies to."""
    return [int(number) for number in re.findall(r"&gt;&gt;(\d+)", post.get("com", ""))]

In [7]:
catalog = requests.get("https://a.4cdn.org/g/catalog.json").json()
live = [thread for thread in catalog[0]["threads"] if not thread.get("sticky")]
busiest = max(live, key=lambda thread: thread["replies"])
resp = requests.get(api_url.format(board="g", op_id=busiest["no"]))
posts = resp.json()["posts"]
len(posts)

279

In [8]:
for post in posts[:12]:
    print(post["no"], "->", quoted_posts(post))

109664794 -> []
109664804 -> [109664794]
109664809 -> [109664804]
109664814 -> [109664809]
109664823 -> [109664814]
109664828 -> [109664823]
109664887 -> [109664828, 109664823]
109664901 -> [109664887]
109664906 -> [109664794]
109664976 -> [109664906]
109665010 -> [109664794]
109665177 -> [109664794, 109664976]


An empty list means the post replies to the thread as a whole. `>>` links can also point to other threads or other boards, so a quoted number is not always in the same thread.